In [1]:
import os
import pandas as pd
import numpy as np

# Read in Dataframes

In [2]:
# individual consumer data
consumers = pd.read_parquet("/uss/hdsi-prismdata/q2-ucsd-consDF.pqt")
consumers.head()

,prism_consumer_id,evaluation_date,credit_score,DQ_TARGET
0,0,2021-09-01,726.0,0.0
1,1,2021-07-01,626.0,0.0
2,2,2021-05-01,680.0,0.0
3,3,2021-03-01,734.0,0.0
4,4,2021-10-01,676.0,0.0


In [3]:
# account information
accounts = pd.read_parquet("/uss/hdsi-prismdata/q2-ucsd-acctDF.pqt")
accounts.head()

,prism_consumer_id,prism_account_id,account_type,balance_date,balance
0,3023,0,SAVINGS,2021-08-31,90.57
1,3023,1,CHECKING,2021-08-31,225.95
2,4416,2,SAVINGS,2022-03-31,15157.17
3,4416,3,CHECKING,2022-03-31,66.42
4,4227,4,CHECKING,2021-07-31,7042.90


In [4]:
# transactional data
transactions = pd.read_parquet("/uss/hdsi-prismdata/q2-ucsd-trxnDF.pqt")
transactions.head()

,prism_consumer_id,prism_transaction_id,category,amount,credit_or_debit,posted_date
0,3023,0,4,0.05,CREDIT,2021-04-16
1,3023,1,12,481.56,CREDIT,2021-04-30
2,3023,2,4,0.05,CREDIT,2021-05-16
3,3023,3,4,0.07,CREDIT,2021-06-16
4,3023,4,4,0.06,CREDIT,2021-07-16


In [5]:
# category mapping
category_mapping = pd.read_csv("/uss/hdsi-prismdata/q2-ucsd-cat-map.csv")
category_mapping.head()

,category_id,category
0,0,SELF_TRANSFER
1,1,EXTERNAL_TRANSFER
2,2,DEPOSIT
3,3,PAYCHECK
4,4,MISCELLANEOUS


In [6]:
# unique categories
accounts['account_type'].unique()

array(['SAVINGS', 'CHECKING', 'CREDIT CARD', 'LINE OF CREDIT',
       'MONEYMARKET', 'LOAN', 'MONEY MARKET', 'ROTH', 'MORTGAGE',
       'RETIREMENT', 'PREPAID', 'BROKERAGE', 'CONSUMER', 'CD', 'IRA',
       'AUTO', 'STUDENT', 'HSA', 'CASH MANAGEMENT', 'OTHER', '401K',
       'STOCK PLAN', 'OVERDRAFT', 'HOME EQUITY'], dtype=object)

# Cashflow Feature Engineering: Yearly
Create features to track yearly income across accounts.

In [7]:
# datetime fix
transactions['posted_date'] = pd.to_datetime(transactions['posted_date'], errors='coerce')
transactions['month_period'] = transactions['posted_date'].dt.to_period('M')

In [9]:
import pandas as pd

income_categories = [2, 3, 6, 7, 8, 9]

# ensure datetime
consumers["evaluation_date"] = pd.to_datetime(consumers["evaluation_date"])
transactions["posted_date"] = pd.to_datetime(transactions["posted_date"])

# filter to income transactions
income_txn = transactions[
    (transactions["credit_or_debit"].str.upper() == "CREDIT") &
    (transactions["category"].isin(income_categories))
].copy()

# attach evaluation_date
income_txn = income_txn.merge(
    consumers[["prism_consumer_id", "evaluation_date"]],
    on="prism_consumer_id",
    how="inner",
)

# 365-day lookback window
income_txn = income_txn[
    (income_txn["posted_date"] <= income_txn["evaluation_date"]) &
    (income_txn["posted_date"] >= income_txn["evaluation_date"] - pd.Timedelta(days=365))
]

# aggregate into new dataframe
yearly_income_df = (
    income_txn.groupby("prism_consumer_id", as_index=False)
    .agg(yearly_income_365d=("amount", "sum"))
)

# fill 0 for no transactions
yearly_income_df["yearly_income_365d"] = yearly_income_df["yearly_income_365d"].fillna(0.0)
yearly_income_df

,prism_consumer_id,yearly_income_365d
0,0,7590.22
1,1,12455.54
2,10,14198.13
3,10000,68037.83
4,10001,42074.14
...,...,...
13250,9995,11226.84
13251,9996,0.03
13252,9997,17007.86
13253,9998,8131.86


# Model Evaluation
Use LogReg to evaluate yearly income feature on accuracy, ROC/AUC, and KS statistic.

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from scipy.stats import ks_2samp

evaluation_df = yearly_income_df.merge(consumers[['prism_consumer_id', 'DQ_TARGET']], on='prism_consumer_id')
evaluation_df = evaluation_df.dropna(subset=["DQ_TARGET", "yearly_income_365d"])

X = evaluation_df[['yearly_income_365d']]
y = evaluation_df['DQ_TARGET']

log_reg = LogisticRegression(max_iter=2000)
log_reg.fit(X, y)

y_pred_proba = log_reg.predict_proba(X)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

acc = accuracy_score(y, y_pred)
auc = roc_auc_score(y, y_pred_proba)

ks, _ = ks_2samp(
    y_pred_proba[y == 1],
    y_pred_proba[y == 0]
)

print(f"Accuracy: {acc:.4f}")
print(f"AUC: {auc:.4f}")
print(f"KS: {ks:.4f}")

Accuracy: 0.9143
AUC: 0.5962
KS: 0.1748
